In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Tubo

## Install additional libraries

In [ ]:
!pip install hydra-core segmentation_models_pytorch==0.3.3 --no-index --find-links=/kaggle/input/ex-library

In [ ]:
# %cd /kaggle/input/cmi-code/kaggle-child-mind-institute-detect-sleep-states/
%cd /kaggle/input/zzzs-tubo/CMI-tubo/

## Config

In [ ]:
# Config
DURATION = 5760
DOWNSAMPLE_RATE = 2
PHASE = 'test'
EXP_NAME = 'exp001'

## Preprocess

In [ ]:
!python -m run.prepare_data dir=kaggle phase=$PHASE

## Inference

In [ ]:
!python -m run.inference\
    dir=kaggle\
    model.encoder_name=resnet34\
    model.encoder_weights=null\
    num_workers=2\
    exp_name=$EXP_NAME\
    weight.run_name=single\
    batch_size=64\
    duration=$DURATION\
    downsample_rate=$DOWNSAMPLE_RATE\
    post_process.score_th=0.005\
    post_process.distance=40\
    phase=$PHASE

In [ ]:
# いらないファイル、フォルダ削除
!rm -rf /kaggle/working/processed_data
!rm -rf /kaggle/working/output

# GRUNETB - 2f

In [ ]:
!python /kaggle/input/zzzs-grunet/GRUNETB-2f/inference.py

In [ ]:
! rm -r /kaggle/working/series-grunet/

# Ensemble predictions

In [ ]:
PROB_LOC1 = "/kaggle/working/grunetb-2f-preds/"
PROB_LOC2 = "/kaggle/working/cmi-tubo-preds/"

In [ ]:
# Collect GRUNET preds

probs1_df = pd.DataFrame()
serieslenDict = dict()

for file in os.listdir(PROB_LOC1):
    
    series_id = file.split(".csv")[0]
    
    path = PROB_LOC1 + file
    temp = pd.read_csv(path)
    temp['series_id'] = series_id
    
    serieslenDict[series_id] = len(temp)
    
    probs1_df = pd.concat([probs1_df, temp], ignore_index=True)

probs1_df = probs1_df.reset_index(drop=True)
probs1_df.rename(columns={'onset':'onset_1',
                         'wakeup':'wakeup_1'}, inplace=True)
probs1_df.shape

In [ ]:
# Collect Tubo preds

probs2_df = pd.DataFrame()

for file in os.listdir(PROB_LOC2):
    
    series_id = file.split(".npy")[0]
    predlen = serieslenDict[series_id]
    
    path = PROB_LOC2 + file
    temp_ = np.load(path)
    
    temp = pd.DataFrame()
    temp["onset_2"] = temp_[:predlen, 0]
    temp["wakeup_2"] = temp_[:predlen, 1]
    temp["step"] = range(0,predlen)
    temp["series_id"] = series_id
    
    probs2_df = pd.concat([probs2_df, temp], ignore_index=True)

probs2_df = probs2_df.reset_index(drop=True)
probs2_df.shape

In [ ]:
probs_df = pd.merge(probs1_df, probs2_df, on=["series_id","step"], how="left")

print(probs_df.shape)
probs_df.head(2)

In [ ]:
# Fields to normalize
fields_to_normalize = ['wakeup_1', 'onset_1', 'wakeup_2', 'onset_2']

# Normalizing within each series_id
for field in fields_to_normalize:
    probs_df[f'{field}_norm'] = probs_df.groupby('series_id')[field].transform(lambda x: (x - x.min()) / (x.max() - x.min()))

# Simple averaging of normalized probabilities
# probs_df['wakeup_avg'] = (probs_df['wakeup_1_norm'] + probs_df['wakeup_2_norm']) / 2
# probs_df['onset_avg'] = (probs_df['onset_1_norm'] + probs_df['onset_2_norm']) / 2

# Weighted averaging
# Adjust weights based on model performance
weight_1 = 0.3 #grunetb-2f
weight_2 = 0.7 #tubo
total_weight = weight_1 + weight_2

probs_df['wakeup_avg'] = (probs_df['wakeup_1_norm'] * weight_1 + probs_df['wakeup_2_norm'] * weight_2) / total_weight
probs_df['onset_avg'] = (probs_df['onset_1_norm'] * weight_1 + probs_df['onset_2_norm'] * weight_2) / total_weight

# Drop all probabilities except the averaged ones
probs_df.drop(columns=['wakeup_1', 'wakeup_2', 'onset_1', 'onset_2', 'wakeup_1_norm', 'wakeup_2_norm', 'onset_1_norm', 'onset_2_norm'], inplace=True)

# Rename columns
probs_df.rename(columns={'wakeup_avg': 'wakeup', 
                         'onset_avg': 'onset'}, inplace=True)

probs_df = probs_df[['series_id', 'step', 'wakeup', 'onset']]
probs_df.shape

# Final Prediction logic

In [ ]:
from scipy.signal import find_peaks

seriesLst = probs_df['series_id'].unique().tolist()

SIGMA = 36
score_th = 0.001
distance = 5000


In [ ]:
records = []
for series_id in tqdm(seriesLst):
    
    series_df = probs_df[probs_df['series_id'] == series_id]
    series_df = series_df.reset_index(drop=True)

    for i, event_name in enumerate(["onset", "wakeup"]):
        
#         score_th = series_df[event_name].max() * 0.1
        
        steps = find_peaks(series_df[event_name], height=score_th, distance=distance)[0]
        scores = series_df.loc[steps, event_name].values

        for step, score in zip(steps, scores):
            records.append(
                {
                    "series_id": series_id,
                    "step": step,
                    "event": event_name,
                    "score": score,
                }
            )

    if len(records) == 0:  # 一つも予測がない場合はdummyを入れる
        records.append(
            {
                "series_id": series_id,
                "step": 0,
                "event": "onset",
                "score": 0,
            }
        )



# Submission

In [ ]:
preds_df = pd.DataFrame(records)
preds_df = preds_df.reset_index(drop=True)
preds_df = preds_df.sort_values(["series_id", "step"]).reset_index(drop=True)
preds_df['row_id'] = np.arange(len(preds_df))
preds_df = preds_df[['row_id', 'series_id', 'step', 'event', 'score']]

In [ ]:
preds_df.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
preds_df.head()